# CREMA-D Speech Emotion Recognition Pipeline

This notebook runs the cleaned pipeline from `crema_ser_pipeline.py`: rebuilt paths, cached MFCC features, actor-aware evaluation, and confidence-ranked predictions.

In [1]:
from pathlib import Path

from crema_ser_pipeline import (
    build_or_clean_metadata,
    load_or_extract_features,
    train_and_evaluate,
    save_artifacts,
)

In [2]:
# Change these if your files are stored somewhere else.
METADATA_CSV = Path.home() / "Downloads" / "crema_metadata (1).csv"
AUDIO_DIR = Path.home() / "Downloads" / "archive" / "Crema"
OUTPUT_DIR = Path("artifacts")

SR = 16000
N_MFCC = 40
TRIM_TOP_DB = 30
N_JOBS = -1

In [3]:
metadata = build_or_clean_metadata(
    metadata_csv=METADATA_CSV,
    audio_dir=AUDIO_DIR,
    output_dir=OUTPUT_DIR,
)
metadata.head()

Saved cleaned metadata: /Users/jordanacle/Documents/Playground 2/artifacts/crema_metadata_clean.csv
Valid audio files: 6004/6004 under /Users/jordanacle/Downloads/archive/Crema


,path,actor_id,emotion,emo_code,filename,path_exists
0,/Users/jordanacle/Downloads/archive/Crema/1001...,1001,angry,ANG,1001_IEO_ANG_HI.wav,True
1,/Users/jordanacle/Downloads/archive/Crema/1001...,1001,happy,HAP,1001_DFA_HAP_XX.wav,True
2,/Users/jordanacle/Downloads/archive/Crema/1001...,1001,angry,ANG,1001_IEO_ANG_LO.wav,True
3,/Users/jordanacle/Downloads/archive/Crema/1001...,1001,fear,FEA,1001_IEO_FEA_LO.wav,True
4,/Users/jordanacle/Downloads/archive/Crema/1001...,1001,fear,FEA,1001_DFA_FEA_XX.wav,True


In [4]:
metadata["path_exists"].value_counts(dropna=False)

path_exists
True    6004
Name: count, dtype: int64

In [5]:
X, y, groups, filenames, feature_metadata = load_or_extract_features(
    metadata,
    output_dir=OUTPUT_DIR,
    sr=SR,
    n_mfcc=N_MFCC,
    trim_top_db=TRIM_TOP_DB,
    n_jobs=N_JOBS,
)
X.shape, y.shape, groups.shape

Extracting MFCC features for 6004 files with n_jobs=-1...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done 632 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed:    4.7s
[Parallel(n_jobs=-1)]: Done 1136 tasks      | elapsed:    5.6s
[Parallel(n_jobs=-1)]: Done 1442 tasks      | elapsed:    6.6s
[Parallel(n_jobs=-1)]: Done 1784 tasks      | elapsed:    7.7s
[Parallel(n_jobs=-1)]: Done 2162 tasks      | elapsed:    9.0s
[Parallel(n_jobs=-1)]: Done 2576 tasks      | elapsed:   10.4s
[Parallel(n_jobs=-1)]: Done 3026 tasks      | elapsed:   12.0s
[Parallel(n_jobs=-1)]: Done 3512 tasks      | elapsed:   13.6s
[Parallel(n_jobs=-1)]: Done 4034 tasks      

Saved features: /Users/jordanacle/Documents/Playground 2/artifacts/X_mfcc.npy (6004 samples, 80 dims)
Feature extraction time: 0.37 minutes


[Parallel(n_jobs=-1)]: Done 6004 out of 6004 | elapsed:   22.2s finished


((6004, 80), (6004,), (6004,))

In [6]:
model, predictions, metrics = train_and_evaluate(
    X=X,
    y=y,
    groups=groups,
    filenames=filenames,
    output_dir=OUTPUT_DIR,
)
save_artifacts(model, OUTPUT_DIR)
metrics["accuracy"], metrics["macro_f1"]

Accuracy: 0.476
Macro F1: 0.472
              precision    recall  f1-score   support

       angry      0.629     0.695     0.661       210
     disgust      0.348     0.374     0.361       206
        fear      0.466     0.415     0.439       212
       happy      0.431     0.402     0.416       219
     neutral      0.449     0.384     0.414       185
         sad      0.511     0.579     0.543       209

    accuracy                          0.476      1241
   macro avg      0.472     0.475     0.472      1241
weighted avg      0.473     0.476     0.473      1241

Saved predictions: /Users/jordanacle/Documents/Playground 2/artifacts/predictions_test.csv
Saved model: /Users/jordanacle/Documents/Playground 2/artifacts/crema_ser_model.joblib


(0.47622884770346496, 0.47214378018089304)

In [7]:
predictions[[
    "filename",
    "true_label",
    "predicted_label",
    "confidence",
    "ranked_class_probabilities",
]].head()

,filename,true_label,predicted_label,confidence,ranked_class_probabilities
0,1001_IEO_ANG_HI.wav,angry,angry,0.724688,"[{""class"": ""angry"", ""probability"": 0.724687717..."
1,1001_DFA_HAP_XX.wav,happy,happy,0.553764,"[{""class"": ""happy"", ""probability"": 0.553764403..."
2,1001_IEO_ANG_LO.wav,angry,fear,0.490290,"[{""class"": ""fear"", ""probability"": 0.4902904740..."
3,1001_IEO_FEA_LO.wav,fear,fear,0.564341,"[{""class"": ""fear"", ""probability"": 0.5643414034..."
4,1001_DFA_FEA_XX.wav,fear,happy,0.441143,"[{""class"": ""happy"", ""probability"": 0.441142983..."


In [8]:
# Re-running this cell should load cached features unless metadata, audio files,
# or feature settings changed.
print(f"Artifacts saved to: {OUTPUT_DIR.resolve()}")

Artifacts saved to: /Users/jordanacle/Documents/Playground 2/artifacts
